# 02 — Deepnote Confirmatory Analysis + Visualisations

This notebook preserves the preregistered confirmatory analyses and adds a separate visualisation layer.

Confirmatory analyses:
- Study 1 OLS with model×scenario fixed effects + HC2 robust SEs;
- Study 1 ±2-point equivalence test;
- Study 2 Turn-5 ANCOVA controlling for Turn-1 outcome + model×scenario fixed effects + HC2;
- analogous legitimacy and recommendation−legitimacy gap models;
- preregistered change-score and five-turn trajectory robustness checks;
- model/category results treated as descriptive heterogeneity.

Visualisations are descriptive presentations of these same outcomes; they do not change the confirmatory specification.

In [ ]:
%pip install -q pandas numpy scipy statsmodels matplotlib

In [ ]:
import os, sys, json, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

# Robust Deepnote loader for llm_experiment_core.py
candidates = [
    Path('/datasets/_deepnote_work/llm_experiment_core.py'),
    Path('/work/llm_experiment_core.py'),
    Path.cwd() / 'llm_experiment_core.py',
]
core_path = next((p for p in candidates if p.exists()), None)
if core_path is None:
    raise FileNotFoundError('Could not find llm_experiment_core.py in the Deepnote Files area.')

spec = importlib.util.spec_from_file_location('llm_experiment_core', core_path)
llm_experiment_core = importlib.util.module_from_spec(spec)
sys.modules['llm_experiment_core'] = llm_experiment_core
spec.loader.exec_module(llm_experiment_core)
globals().update({n: getattr(llm_experiment_core, n) for n in dir(llm_experiment_core) if not n.startswith('_')})

ROOT = locate_project_root()
RAW = ROOT / 'raw'
EXPORTS = ROOT / 'analysis_exports'
FIGURES = EXPORTS / 'figures'
EXPORTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
manifest, SCENARIOS, _ = load_manifest()

MODEL_LABELS = {k: v['display_name'] for k, v in MODEL_REGISTRY.items()}
print('Loaded:', core_path)
print('Raw data:', RAW)
print('Figure exports:', FIGURES)

In [ ]:
def load_confirmatory():
    raw_files = [RAW / "study1_confirmatory.jsonl", RAW / "study2_confirmatory.jsonl"]
    if all(p.exists() for p in raw_files):
        return tuple(pd.DataFrame(read_jsonl(p)) for p in raw_files)
    data_dir = core_path.parent / "data"
    return (pd.read_csv(data_dir / "study1_analysis_ready.csv.gz"),
            pd.read_csv(data_dir / "study2_analysis_ready_long.csv.gz"))

s1, s2 = load_confirmatory()
print("Study 1 rows:", len(s1))
print("Study 2 turn rows:", len(s2))


In [ ]:
# Exact sample audit
assert s1["slot_id"].nunique() == 2940
assert s2["slot_id"].nunique() == 2940
assert len(s1) == 2940
assert len(s2) == 14700
assert s1.groupby(["model_key","scenario_id","condition"])["slot_id"].nunique().eq(15).all()
assert s2.groupby(["model_key","scenario_id","condition"])["slot_id"].nunique().eq(15).all()
assert s2.groupby("slot_id")["turn"].nunique().eq(5).all()
print("PASS: collection structure matches preregistration.")

In [ ]:
# Plot helpers

def mean_ci(df, group_cols, outcome):
    g = df.groupby(group_cols)[outcome]
    out = g.agg(['mean', 'count', 'std']).reset_index()
    out['se'] = out['std'] / np.sqrt(out['count'])
    out['ci95'] = 1.96 * out['se']
    return out


def savefig(name):
    png = FIGURES / f'{name}.png'
    pdf = FIGURES / f'{name}.pdf'
    plt.savefig(png, dpi=300, bbox_inches='tight')
    plt.savefig(pdf, bbox_inches='tight')
    print('Saved:', png)
    print('Saved:', pdf)


def treatment_coef(model, term):
    ci = model.conf_int().loc[term]
    return {
        'estimate': float(model.params[term]),
        'low': float(ci.iloc[0]),
        'high': float(ci.iloc[1]),
        'p': float(model.pvalues[term]),
    }


## Study 1 — primary recommendation effect and equivalence

In [ ]:
s1 = s1.copy()
s1["directional"] = (s1["condition"]=="directional").astype(int)

m1 = smf.ols(
    "recommendation ~ directional + C(model_key):C(scenario_id)",
    data=s1
).fit(cov_type="HC2", use_t=True)
print(m1.summary().tables[1])

b = float(m1.params["directional"])
se = float(m1.bse["directional"])
df = float(m1.df_resid)
delta = 2.0

# TOST: H01 beta <= -2; H02 beta >= +2
t_lower = (b - (-delta))/se
p_lower = 1 - stats.t.cdf(t_lower, df)
t_upper = (b - delta)/se
p_upper = stats.t.cdf(t_upper, df)
ci90 = (b - stats.t.ppf(.95,df)*se, b + stats.t.ppf(.95,df)*se)

print({"beta":b, "SE_HC2":se, "CI90":ci90, "TOST_p_lower":p_lower, "TOST_p_upper":p_upper,
       "equivalent_within_±2": (p_lower<.05 and p_upper<.05)})

In [ ]:
# Secondary Study 1 legitimacy model
m1_legit = smf.ols(
    "legitimacy ~ directional + C(model_key):C(scenario_id)",
    data=s1
).fit(cov_type="HC2", use_t=True)
print(m1_legit.summary().tables[1])

### Visualisation — Study 1 pooled outcomes and treatment effects

The left panels show raw condition means with 95% confidence intervals. The right panel shows the fixed-effect regression treatment estimates with 95% confidence intervals.

In [ ]:
# Study 1 raw means
s1_means_rec = mean_ci(s1, ['condition'], 'recommendation')
s1_means_leg = mean_ci(s1, ['condition'], 'legitimacy')

display(s1_means_rec)
display(s1_means_leg)

# Figure 1A: recommendation means
plt.figure(figsize=(6,4))
x = np.arange(len(s1_means_rec))
plt.errorbar(x, s1_means_rec['mean'], yerr=s1_means_rec['ci95'], fmt='o', capsize=4)
plt.xticks(x, s1_means_rec['condition'].str.title())
plt.ylabel('Recommendation (0–100)')
plt.title('Study 1: recommendation by framing condition')
plt.ylim(0,100)
plt.grid(axis='y', alpha=.2)
savefig('study1_recommendation_means')
plt.show()

# Figure 1B: legitimacy means
plt.figure(figsize=(6,4))
x = np.arange(len(s1_means_leg))
plt.errorbar(x, s1_means_leg['mean'], yerr=s1_means_leg['ci95'], fmt='o', capsize=4)
plt.xticks(x, s1_means_leg['condition'].str.title())
plt.ylabel('Legitimacy (0–100)')
plt.title('Study 1: legitimacy by framing condition')
plt.ylim(0,100)
plt.grid(axis='y', alpha=.2)
savefig('study1_legitimacy_means')
plt.show()

# Figure 1C: regression effects
fx = pd.DataFrame([
    {'Outcome':'Recommendation', **treatment_coef(m1, 'directional')},
    {'Outcome':'Legitimacy', **treatment_coef(m1_legit, 'directional')},
])
fx['y'] = np.arange(len(fx))
plt.figure(figsize=(7,3.5))
plt.axvline(0, linewidth=1)
plt.errorbar(fx['estimate'], fx['y'],
             xerr=[fx['estimate']-fx['low'], fx['high']-fx['estimate']],
             fmt='o', capsize=4)
plt.yticks(fx['y'], fx['Outcome'])
plt.xlabel('Directional-framing effect (points)')
plt.title('Study 1: adjusted treatment effects')
plt.grid(axis='x', alpha=.2)
savefig('study1_adjusted_effects')
plt.show()

# Figure 1D: equivalence view for recommendation
plt.figure(figsize=(7,2.8))
plt.axvline(-2, linestyle='--', linewidth=1)
plt.axvline(0, linewidth=1)
plt.axvline(2, linestyle='--', linewidth=1)
plt.errorbar([b], [0], xerr=[[b-ci90[0]],[ci90[1]-b]], fmt='o', capsize=4)
plt.yticks([0], ['Recommendation'])
plt.xlabel('Directional-framing effect (90% CI)')
plt.title('Study 1 equivalence test: preregistered ±2-point bounds')
plt.grid(axis='x', alpha=.2)
savefig('study1_equivalence')
plt.show()

### Exploratory Study 1 analyses by model

This additional analysis estimates recommendation and legitimacy effects separately for each model, using scenario fixed effects and HC2 robust standard errors. Recommendation equivalence tests use the same ±2-point margin as the pooled analysis, but these model-specific tests were not preregistered. All p-values are unadjusted for multiple testing. The appendix bar chart shows treatment estimates with 95% confidence intervals; equivalence decisions use 90% intervals.

Holm-adjusted p-values and equivalence results are provided in the later multiple-testing section.


In [ ]:
# Exploratory Study 1 analyses by model.
# These were added after the pooled analyses; p-values are unadjusted.
# Requires s1, MODEL_REGISTRY, MODEL_LABELS, EXPORTS, FIGURES and savefig.
from IPython.display import display

model_order_s1 = [key for key in MODEL_REGISTRY if key in s1['model_key'].unique()]
model_results_s1 = []
for model_key_s1 in model_order_s1:
    model_data_s1 = s1.loc[s1['model_key'].eq(model_key_s1)].copy()
    model_data_s1['directional'] = model_data_s1['condition'].eq('directional').astype(int)
    assert set(model_data_s1['condition']) == {'neutral', 'directional'}
    for outcome_s1 in ['recommendation', 'legitimacy']:
        fit_s1 = smf.ols(
            f'{outcome_s1} ~ directional + C(scenario_id)', data=model_data_s1
        ).fit(cov_type='HC2', use_t=True)
        effect_s1 = float(fit_s1.params['directional'])
        se_s1 = float(fit_s1.bse['directional'])
        df_s1 = float(fit_s1.df_resid)
        ci95_s1 = fit_s1.conf_int(alpha=.05).loc['directional']
        ci90_s1 = fit_s1.conf_int(alpha=.10).loc['directional']
        result_s1 = {
            'model_key': model_key_s1,
            'model_label': MODEL_LABELS[model_key_s1],
            'outcome': outcome_s1,
            'n': int(fit_s1.nobs),
            'n_neutral': int(model_data_s1['condition'].eq('neutral').sum()),
            'n_directional': int(model_data_s1['condition'].eq('directional').sum()),
            'neutral_mean': float(model_data_s1.loc[model_data_s1['directional'].eq(0), outcome_s1].mean()),
            'directional_mean': float(model_data_s1.loc[model_data_s1['directional'].eq(1), outcome_s1].mean()),
            'estimate': effect_s1, 'se_HC2': se_s1,
            'ci95_low': float(ci95_s1.iloc[0]), 'ci95_high': float(ci95_s1.iloc[1]),
            'p_unadjusted': float(fit_s1.pvalues['directional']),
            'ci90_low': float(ci90_s1.iloc[0]), 'ci90_high': float(ci90_s1.iloc[1]),
            'equivalence_margin': np.nan,
            'TOST_p_lower_unadjusted': np.nan, 'TOST_p_upper_unadjusted': np.nan,
            'TOST_p_unadjusted': np.nan, 'equivalent_within_margin': None,
        }
        # Apply the pooled ±2-point margin to recommendation only.
        # Model-specific equivalence tests are exploratory, not preregistered tests.
        if outcome_s1 == 'recommendation':
            margin_s1 = 2.0
            p_lower_s1 = float(stats.t.sf((effect_s1 + margin_s1) / se_s1, df_s1))
            p_upper_s1 = float(stats.t.cdf((effect_s1 - margin_s1) / se_s1, df_s1))
            result_s1.update({
                'equivalence_margin': margin_s1,
                'TOST_p_lower_unadjusted': p_lower_s1,
                'TOST_p_upper_unadjusted': p_upper_s1,
                'TOST_p_unadjusted': max(p_lower_s1, p_upper_s1),
                'equivalent_within_margin': bool(p_lower_s1 < .05 and p_upper_s1 < .05),
            })
        model_results_s1.append(result_s1)

study1_by_model = pd.DataFrame(model_results_s1)
model_csv_s1 = EXPORTS / 'exploratory_study1_by_model.csv'
study1_by_model.to_csv(model_csv_s1, index=False)
print('Exploratory analyses: scenario fixed effects, HC2 standard errors, t-based inference.')
print('All p-values are unadjusted for multiple testing. Equivalence uses the 90% CI, not the plotted 95% CI.')
display(study1_by_model[['model_label', 'outcome', 'n', 'estimate', 'ci95_low', 'ci95_high',
                       'p_unadjusted', 'ci90_low', 'ci90_high', 'TOST_p_unadjusted',
                       'equivalent_within_margin']].round(4))

# Appendix bar chart: treatment effects rather than raw condition means.
# All models have the same scenario composition; bars are in the same order.
fig_s1, axes_s1 = plt.subplots(1, 2, figsize=(12, 5.8), sharey=True)
y_s1 = np.arange(len(model_order_s1))
for ax_s1, outcome_s1, title_s1 in zip(
    axes_s1, ['recommendation', 'legitimacy'], ['Recommendation', 'Perceived legitimacy']
):
    plot_s1 = study1_by_model.loc[study1_by_model['outcome'].eq(outcome_s1)].set_index('model_key').reindex(model_order_s1)
    ax_s1.barh(y_s1, plot_s1['estimate'], height=.55, color='#466B8A', alpha=.85)
    ax_s1.errorbar(plot_s1['estimate'], y_s1,
                  xerr=np.vstack([plot_s1['estimate'] - plot_s1['ci95_low'],
                                  plot_s1['ci95_high'] - plot_s1['estimate']]),
                  fmt='none', ecolor='black', capsize=3, linewidth=1.1)
    ax_s1.axvline(0, color='black', linewidth=.8)
    ax_s1.set_title(title_s1)
    ax_s1.set_xlabel('Initial interest − neutral request (points)')
    ax_s1.grid(axis='x', alpha=.15)
    ax_s1.set_axisbelow(True)
    ax_s1.spines[['top', 'right']].set_visible(False)
axes_s1[0].axvline(-2, color='#777777', linestyle='--', linewidth=1)
axes_s1[0].axvline(2, color='#777777', linestyle='--', linewidth=1)
# Common scale makes the two outcome panels directly comparable.
limit_s1 = max(2.25, abs(study1_by_model[['ci95_low', 'ci95_high']].to_numpy()).max() * 1.12)
for ax_s1 in axes_s1:
    ax_s1.set_xlim(-limit_s1, limit_s1)
axes_s1[0].set_yticks(y_s1, [MODEL_LABELS[k] for k in model_order_s1])
axes_s1[0].invert_yaxis()
fig_s1.suptitle('Study 1: exploratory treatment effects by model', fontsize=13)
fig_s1.text(.02, .025, 'Bars: OLS estimates with scenario fixed effects. Error bars: 95% CIs using HC2 standard errors.\n'
            'Dashed lines: ±2-point recommendation margin. Equivalence is assessed separately using 90% CIs. N = 420 per model.',
            fontsize=9)
fig_s1.tight_layout(rect=[0, .12, 1, .95])
savefig('exploratory_study1_treatment_effects_by_model')
plt.show()
print('Saved:', model_csv_s1)


## Study 2 — primary Turn-5 ANCOVA

In [ ]:
s2 = s2.copy()
s2["pressure"] = (s2["condition"]=="pressure").astype(int)

wide = s2.pivot_table(
    index=["slot_id","model_key","scenario_id","intended_category","condition","pressure"],
    columns="turn",
    values=["recommendation","legitimacy"],
    aggfunc="first"
)
wide.columns = [f"{a}_T{b}" for a,b in wide.columns]
wide = wide.reset_index()

m2 = smf.ols(
    "recommendation_T5 ~ pressure + recommendation_T1 + C(model_key):C(scenario_id)",
    data=wide
).fit(cov_type="HC2", use_t=True)
print(m2.summary().tables[1])

## Study 2 — legitimacy and recommendation−legitimacy gap

In [ ]:
m2_legit = smf.ols(
    "legitimacy_T5 ~ pressure + legitimacy_T1 + C(model_key):C(scenario_id)",
    data=wide
).fit(cov_type="HC2", use_t=True)

wide["gap_T1"] = wide["recommendation_T1"] - wide["legitimacy_T1"]
wide["gap_T5"] = wide["recommendation_T5"] - wide["legitimacy_T5"]

m2_gap = smf.ols(
    "gap_T5 ~ pressure + gap_T1 + C(model_key):C(scenario_id)",
    data=wide
).fit(cov_type="HC2", use_t=True)

print("LEGITIMACY")
print(m2_legit.summary().tables[1])
print("\nGAP")
print(m2_gap.summary().tables[1])

## Robustness 1 — T1→T5 change scores

In [ ]:
wide["d_recommendation"] = wide["recommendation_T5"] - wide["recommendation_T1"]
wide["d_legitimacy"] = wide["legitimacy_T5"] - wide["legitimacy_T1"]

change_rec = smf.ols(
    "d_recommendation ~ pressure + C(model_key):C(scenario_id)",
    data=wide
).fit(cov_type="HC2", use_t=True)
change_leg = smf.ols(
    "d_legitimacy ~ pressure + C(model_key):C(scenario_id)",
    data=wide
).fit(cov_type="HC2", use_t=True)

print(change_rec.summary().tables[1])
print(change_leg.summary().tables[1])

## Robustness 2 — full five-turn trajectory

In [ ]:
# Turn treated categorically; interaction terms describe how the randomized arms diverge over time.
traj = smf.ols(
    "recommendation ~ pressure * C(turn) + C(model_key):C(scenario_id)",
    data=s2
).fit(
    cov_type="cluster",
    cov_kwds={"groups": s2["slot_id"]},
    use_t=True
)
print(traj.summary().tables[1])

### Visualisation — Study 2 five-turn trajectories

These plots show raw mean trajectories with 95% confidence intervals. The randomized contrast is the pressure trajectory versus neutral reassessment; turn-by-turn differences are descriptive dynamics.

In [ ]:
# Add gap at the turn level for plotting
s2['gap'] = s2['recommendation'] - s2['legitimacy']

for outcome, ylabel, filename in [
    ('recommendation', 'Recommendation (0–100)', 'study2_recommendation_trajectory'),
    ('legitimacy', 'Legitimacy (0–100)', 'study2_legitimacy_trajectory'),
    ('gap', 'Recommendation − legitimacy', 'study2_gap_trajectory'),
]:
    p = mean_ci(s2, ['condition','turn'], outcome)
    plt.figure(figsize=(7.5,4.5))
    for cond, g in p.groupby('condition'):
        plt.errorbar(g['turn'], g['mean'], yerr=g['ci95'], marker='o', capsize=3, label=cond.title())
    plt.xticks([1,2,3,4,5])
    plt.xlabel('Conversation turn')
    plt.ylabel(ylabel)
    plt.title(f'Study 2: {ylabel.lower()} across turns')
    if outcome in ['recommendation','legitimacy']:
        plt.ylim(0,100)
    plt.legend(frameon=False)
    plt.grid(axis='y', alpha=.2)
    savefig(filename)
    plt.show()

# Adjusted Turn-5 treatment effects
fx2 = pd.DataFrame([
    {'Outcome':'Recommendation', **treatment_coef(m2, 'pressure')},
    {'Outcome':'Legitimacy', **treatment_coef(m2_legit, 'pressure')},
    {'Outcome':'Rec. − legitimacy gap', **treatment_coef(m2_gap, 'pressure')},
])
fx2['y'] = np.arange(len(fx2))
plt.figure(figsize=(7.5,4))
plt.axvline(0, linewidth=1)
plt.errorbar(fx2['estimate'], fx2['y'],
             xerr=[fx2['estimate']-fx2['low'], fx2['high']-fx2['estimate']],
             fmt='o', capsize=4)
plt.yticks(fx2['y'], fx2['Outcome'])
plt.xlabel('Pressure effect at Turn 5 (points)')
plt.title('Study 2: adjusted Turn-5 treatment effects')
plt.grid(axis='x', alpha=.2)
savefig('study2_adjusted_turn5_effects')
plt.show()

## Prespecified descriptive heterogeneity

In [ ]:
t5 = s2[s2["turn"]==5].copy()

model_desc = t5.groupby(["model_key","condition"])[["recommendation","legitimacy"]].mean().unstack("condition")
category_desc = t5.groupby(["intended_category","condition"])[["recommendation","legitimacy"]].mean().unstack("condition")

display(model_desc)
display(category_desc)

model_desc.to_csv(EXPORTS/"descriptive_by_model.csv")
category_desc.to_csv(EXPORTS/"descriptive_by_category.csv")

### Visualisation — descriptive heterogeneity

These figures are descriptive, consistent with the preregistration. They are not separately powered confirmatory tests.

In [ ]:
# Model-specific Study 2 Turn-5 means
model_t5 = mean_ci(t5, ['model_key','condition'], 'recommendation')
model_order = list(MODEL_REGISTRY.keys())

plt.figure(figsize=(9,5.5))
for cond, marker in [('neutral','o'),('pressure','s')]:
    g = model_t5[model_t5['condition']==cond].set_index('model_key').reindex(model_order).reset_index()
    x = np.arange(len(g)) + (-0.08 if cond=='neutral' else 0.08)
    plt.errorbar(x, g['mean'], yerr=g['ci95'], fmt=marker, capsize=3, label=cond.title())
plt.xticks(np.arange(len(model_order)), [MODEL_LABELS[k] for k in model_order], rotation=35, ha='right')
plt.ylabel('Turn-5 recommendation (0–100)')
plt.title('Study 2: Turn-5 recommendation by model')
plt.ylim(0,100)
plt.legend(frameon=False)
plt.grid(axis='y', alpha=.2)
savefig('study2_turn5_recommendation_by_model')
plt.show()

# Category-specific Study 2 trajectories
cat = mean_ci(s2, ['intended_category','condition','turn'], 'recommendation')
for category, gcat in cat.groupby('intended_category'):
    plt.figure(figsize=(7,4.2))
    for cond, g in gcat.groupby('condition'):
        plt.errorbar(g['turn'], g['mean'], yerr=g['ci95'], marker='o', capsize=3, label=cond.title())
    plt.xticks([1,2,3,4,5])
    plt.xlabel('Conversation turn')
    plt.ylabel('Recommendation (0–100)')
    plt.ylim(0,100)
    plt.title(f'Study 2: recommendation trajectory — {category}')
    plt.legend(frameon=False)
    plt.grid(axis='y', alpha=.2)
    safe = ''.join(ch if ch.isalnum() else '_' for ch in str(category)).strip('_').lower()
    savefig(f'study2_trajectory_category_{safe}')
    plt.show()

# Scenario-specific Turn-5 pressure minus neutral mean difference
scen = (t5.groupby(['scenario_id','condition'])['recommendation'].mean().unstack('condition'))
scen['difference'] = scen['pressure'] - scen['neutral']
scen = scen.sort_values('difference')
plt.figure(figsize=(8,6))
plt.axvline(0, linewidth=1)
plt.scatter(scen['difference'], np.arange(len(scen)))
plt.yticks(np.arange(len(scen)), scen.index)
plt.xlabel('Pressure − neutral Turn-5 recommendation')
plt.title('Study 2: descriptive scenario-level pressure differences')
plt.grid(axis='x', alpha=.2)
savefig('study2_scenario_pressure_differences')
plt.show()

## Exploratory heterogeneity by scenario category

These analyses were **not preregistered as separate confirmatory tests**. They examine whether the Study 2 pressure effect differs descriptively across the three prespecified scenario categories: legitimate, high-risk legitimate, and scam/fraud-indicative.

Within each category, we estimate the same Turn-5 ANCOVA form used in the confirmatory analysis, separately for recommendation, legitimacy, and the recommendation−legitimacy gap. We report estimates and 95% confidence intervals as exploratory heterogeneity evidence rather than additional confirmatory hypotheses.

In [ ]:
# Exploratory category-specific Turn-5 ANCOVA estimates

def category_ancova_results(wide_df):
    rows = []
    specs = [
        ('Recommendation', 'recommendation_T5', 'recommendation_T1'),
        ('Legitimacy', 'legitimacy_T5', 'legitimacy_T1'),
        ('Recommendation−legitimacy gap', 'gap_T5', 'gap_T1'),
    ]

    for category in sorted(wide_df['intended_category'].dropna().unique()):
        dc = wide_df[wide_df['intended_category'] == category].copy()

        for outcome_label, y, baseline in specs:
            model = smf.ols(
                f'{y} ~ pressure + {baseline} + C(model_key):C(scenario_id)',
                data=dc
            ).fit(cov_type='HC2', use_t=True)

            ci = model.conf_int().loc['pressure']
            rows.append({
                'category': category,
                'outcome': outcome_label,
                'estimate': float(model.params['pressure']),
                'se': float(model.bse['pressure']),
                'ci_low': float(ci.iloc[0]),
                'ci_high': float(ci.iloc[1]),
                'p': float(model.pvalues['pressure']),
                'n_trajectories': int(model.nobs),
            })

    return pd.DataFrame(rows)

category_ancova = category_ancova_results(wide)
display(category_ancova)
category_ancova.to_csv(EXPORTS/'exploratory_category_turn5_ancova.csv', index=False)
print('Saved:', EXPORTS/'exploratory_category_turn5_ancova.csv')

In [ ]:
# Exploratory forest plots: Turn-5 pressure effects by scenario category
for outcome in category_ancova['outcome'].unique():
    d = category_ancova[category_ancova['outcome'] == outcome].copy()
    d = d.sort_values('estimate')
    y = np.arange(len(d))

    plt.figure(figsize=(8, 4.5))
    plt.axvline(0, linewidth=1)
    plt.errorbar(
        d['estimate'], y,
        xerr=[d['estimate'] - d['ci_low'], d['ci_high'] - d['estimate']],
        fmt='o', capsize=4
    )
    plt.yticks(y, d['category'])
    plt.xlabel('Pressure effect at Turn 5 (points)')
    plt.title(f'Exploratory: Turn-5 pressure effect by category — {outcome}')
    plt.grid(axis='x', alpha=.2)

    safe_outcome = ''.join(ch if ch.isalnum() else '_' for ch in outcome).strip('_').lower()
    savefig(f'exploratory_category_turn5_{safe_outcome}')
    plt.show()

In [ ]:
# Exploratory trajectories by category for recommendation, legitimacy, and the gap
# One chart per category/outcome to keep each figure readable.

s2['gap'] = s2['recommendation'] - s2['legitimacy']

trajectory_specs = [
    ('recommendation', 'Recommendation (0–100)', (0, 100)),
    ('legitimacy', 'Legitimacy (0–100)', (0, 100)),
    ('gap', 'Recommendation − legitimacy', None),
]

for category in sorted(s2['intended_category'].dropna().unique()):
    dc = s2[s2['intended_category'] == category].copy()

    for outcome, ylabel, ylim in trajectory_specs:
        p = mean_ci(dc, ['condition', 'turn'], outcome)

        plt.figure(figsize=(7.5, 4.5))
        for cond, g in p.groupby('condition'):
            plt.errorbar(
                g['turn'], g['mean'], yerr=g['ci95'],
                marker='o', capsize=3, label=cond.title()
            )
        plt.xticks([1, 2, 3, 4, 5])
        plt.xlabel('Conversation turn')
        plt.ylabel(ylabel)
        plt.title(f'Exploratory: {ylabel.lower()} — {category}')
        if ylim is not None:
            plt.ylim(*ylim)
        plt.legend(frameon=False)
        plt.grid(axis='y', alpha=.2)

        safe_category = ''.join(ch if ch.isalnum() else '_' for ch in str(category)).strip('_').lower()
        safe_outcome = ''.join(ch if ch.isalnum() else '_' for ch in outcome).strip('_').lower()
        savefig(f'exploratory_{safe_outcome}_trajectory_{safe_category}')
        plt.show()

In [ ]:
# Compact exploratory category table for manuscript drafting
category_table = category_ancova.copy()
category_table['estimate_ci'] = category_table.apply(
    lambda r: f"{r['estimate']:.2f} [{r['ci_low']:.2f}, {r['ci_high']:.2f}]", axis=1
)
category_table_display = category_table.pivot(
    index='category', columns='outcome', values='estimate_ci'
)
display(category_table_display)
category_table_display.to_csv(EXPORTS/'exploratory_category_effects_compact.csv')
print('Saved:', EXPORTS/'exploratory_category_effects_compact.csv')

## Export analysis-ready data and coefficient tables

In [ ]:
# Compact confirmatory summary table
summary = pd.DataFrame([
    {'Study':'1','Outcome':'Recommendation','Term':'Directional framing', **treatment_coef(m1,'directional')},
    {'Study':'1','Outcome':'Legitimacy','Term':'Directional framing', **treatment_coef(m1_legit,'directional')},
    {'Study':'2','Outcome':'Recommendation T5','Term':'Pressure', **treatment_coef(m2,'pressure')},
    {'Study':'2','Outcome':'Legitimacy T5','Term':'Pressure', **treatment_coef(m2_legit,'pressure')},
    {'Study':'2','Outcome':'Recommendation−legitimacy gap T5','Term':'Pressure', **treatment_coef(m2_gap,'pressure')},
])
summary = summary[['Study','Outcome','Term','estimate','low','high','p']]
display(summary)
summary.to_csv(EXPORTS/'confirmatory_summary.csv', index=False)
print('Saved:', EXPORTS/'confirmatory_summary.csv')

In [ ]:
s1.to_csv(EXPORTS/"study1_analysis_ready.csv", index=False)
wide.to_csv(EXPORTS/"study2_analysis_ready_wide.csv", index=False)
s2.to_csv(EXPORTS/"study2_analysis_ready_long.csv", index=False)

def coef_table(model, label):
    ci = model.conf_int()
    return pd.DataFrame({
        "analysis": label,
        "term": model.params.index,
        "estimate": model.params.values,
        "se": model.bse.values,
        "p": model.pvalues.values,
        "ci_low": ci[0].values,
        "ci_high": ci[1].values,
    })

tables = pd.concat([
    coef_table(m1,"study1_recommendation"),
    coef_table(m1_legit,"study1_legitimacy"),
    coef_table(m2,"study2_recommendation_T5_ANCOVA"),
    coef_table(m2_legit,"study2_legitimacy_T5_ANCOVA"),
    coef_table(m2_gap,"study2_gap_T5_ANCOVA"),
    coef_table(change_rec,"study2_change_recommendation"),
    coef_table(change_leg,"study2_change_legitimacy"),
], ignore_index=True)
tables.to_csv(EXPORTS/"confirmatory_coefficients.csv", index=False)
print("Exports written to", EXPORTS)

## Additional exploratory Study 2 analyses by model

Within each model, ANCOVA is estimated by OLS with the outcome’s own Turn 1 score, scenario fixed effects and HC2 robust standard errors. These regressions were added after preregistration. Holm-adjusted p-values are added in the later multiple-testing cell.


In [ ]:
# Additional exploratory Study 2 ANCOVA estimates by model.
# Within each model, scenario fixed effects replace model-by-scenario fixed effects.
study2_model_rows = []
for model_key_s2 in MODEL_REGISTRY:
    dm_s2 = wide.loc[wide['model_key'].eq(model_key_s2)].copy()
    if dm_s2.empty:
        continue
    for outcome_s2, baseline_s2 in [('recommendation_T5', 'recommendation_T1'),
                                    ('legitimacy_T5', 'legitimacy_T1'), ('gap_T5', 'gap_T1')]:
        fit_s2 = smf.ols(f'{outcome_s2} ~ pressure + {baseline_s2} + C(scenario_id)',
                         data=dm_s2).fit(cov_type='HC2', use_t=True)
        ci_s2 = fit_s2.conf_int().loc['pressure']
        study2_model_rows.append({
            'model_key': model_key_s2, 'model_label': MODEL_LABELS[model_key_s2],
            'outcome': outcome_s2, 'baseline': baseline_s2, 'n': int(fit_s2.nobs),
            'estimate': float(fit_s2.params['pressure']), 'se_HC2': float(fit_s2.bse['pressure']),
            'ci95_low': float(ci_s2.iloc[0]), 'ci95_high': float(ci_s2.iloc[1]),
            'p_unadjusted': float(fit_s2.pvalues['pressure']),
        })
study2_by_model = pd.DataFrame(study2_model_rows)
study2_by_model.to_csv(EXPORTS / 'exploratory_study2_by_model.csv', index=False)
display(study2_by_model.round(4))


## Additional descriptive standardised effect sizes

Treatment coefficients are divided by the pooled within-condition standard deviation of the observed dependent variable. Study 2 uses the Turn 5 outcome SD, including a separate SD for the recommendation–legitimacy gap. These are adjusted treatment contrasts in SD units, rather than unadjusted Cohen’s d or residual-standardised effects. Point estimates remain the main reporting scale. Model and category analyses use their respective subgroup SDs. The displayed intervals rescale the regression 95% confidence intervals while treating the observed SD as fixed; they do not account for estimation uncertainty in the denominator. This calculation was added after preregistration.


In [ ]:
# Standardised treatment effects: adjusted coefficient / pooled within-condition SD.
# This is an additional descriptive calculation, not a preregistered analysis.
# SDs refer to observed outcomes, not regression residuals or standard errors.
def standardised_treatment_effect(frame, outcome, condition, coefficient, ci_low, ci_high):
    groups = [pd.to_numeric(g[outcome], errors='raise').dropna()
              for _, g in frame.groupby(condition)]
    if len(groups) != 2 or any(len(g) < 2 for g in groups):
        raise ValueError('Exactly two conditions, with at least two observations each, are required.')
    n_total = sum(len(g) for g in groups)
    pooled_sd = float(np.sqrt(sum((len(g)-1)*g.var(ddof=1) for g in groups) / (n_total-2)))
    if not np.isfinite(pooled_sd) or pooled_sd <= 0:
        raise ValueError(f'Cannot standardise {outcome}: pooled within-condition SD is zero or invalid.')
    return {
        'n': n_total, 'pooled_within_condition_sd': pooled_sd,
        'estimate_points': float(coefficient),
        'ci95_low_points': float(ci_low), 'ci95_high_points': float(ci_high),
        'standardised_effect': float(coefficient) / pooled_sd,
        'ci95_low_sd_units': float(ci_low) / pooled_sd,
        'ci95_high_sd_units': float(ci_high) / pooled_sd,
    }

standardised_rows = []
# The five pooled treatment effects reported in Tables 1 and 2.
pooled_outcome_lookup = {
    ('1', 'Recommendation'): (s1, 'recommendation', 'condition'),
    ('1', 'Legitimacy'): (s1, 'legitimacy', 'condition'),
    ('2', 'Recommendation T5'): (wide, 'recommendation_T5', 'condition'),
    ('2', 'Legitimacy T5'): (wide, 'legitimacy_T5', 'condition'),
    ('2', 'Recommendation−legitimacy gap T5'): (wide, 'gap_T5', 'condition'),
}
for _, row in summary.iterrows():
    study_key = str(row['Study'])
    data_es, outcome_es, condition_es = pooled_outcome_lookup[(study_key, row['Outcome'])]
    standardised_rows.append({
        'scope': 'pooled', 'study': study_key, 'subgroup': 'All models and scenarios',
        'outcome': row['Outcome'], 'term': row['Term'], 'p_unadjusted': float(row['p']),
        **standardised_treatment_effect(data_es, outcome_es, condition_es,
                                      row['estimate'], row['low'], row['high']),
    })

# Additional exploratory Study 1 model-specific effects, if the earlier cell ran.
if 'study1_by_model' in globals():
    for _, row in study1_by_model.iterrows():
        data_es = s1.loc[s1['model_key'].eq(row['model_key'])]
        standardised_rows.append({
            'scope': 'exploratory_model', 'study': '1', 'subgroup': row['model_label'],
            'outcome': row['outcome'], 'term': 'Directional framing',
            'p_unadjusted': float(row['p_unadjusted']),
            **standardised_treatment_effect(data_es, row['outcome'], 'condition',
                                          row['estimate'], row['ci95_low'], row['ci95_high']),
        })

# Additional exploratory Study 2 model-specific effects.
if 'study2_by_model' in globals():
    for _, row in study2_by_model.iterrows():
        data_es = wide.loc[wide['model_key'].eq(row['model_key'])]
        standardised_rows.append({
            'scope': 'exploratory_model', 'study': '2', 'subgroup': row['model_label'],
            'outcome': row['outcome'], 'term': 'Pressure', 'p_unadjusted': float(row['p_unadjusted']),
            **standardised_treatment_effect(data_es, row['outcome'], 'condition',
                                          row['estimate'], row['ci95_low'], row['ci95_high']),
        })

# Exploratory category effects use the SD within the relevant category.
if 'category_ancova' in globals():
    category_outcome_lookup = {
        'Recommendation': 'recommendation_T5', 'Legitimacy': 'legitimacy_T5',
        'Recommendation−legitimacy gap': 'gap_T5',
    }
    for _, row in category_ancova.iterrows():
        data_es = wide.loc[wide['intended_category'].eq(row['category'])]
        standardised_rows.append({
            'scope': 'exploratory_category', 'study': '2', 'subgroup': row['category'],
            'outcome': row['outcome'], 'term': 'Pressure', 'p_unadjusted': float(row['p']),
            **standardised_treatment_effect(data_es, category_outcome_lookup[row['outcome']],
                                          'condition', row['estimate'], row['ci_low'], row['ci_high']),
        })

standardised_effect_sizes = pd.DataFrame(standardised_rows)
standardised_effect_sizes.to_csv(EXPORTS / 'standardised_treatment_effect_sizes.csv', index=False)
pooled_standardised_effect_sizes = standardised_effect_sizes.loc[standardised_effect_sizes['scope'].eq('pooled')]
pooled_standardised_effect_sizes.to_csv(EXPORTS / 'pooled_standardised_treatment_effect_sizes.csv', index=False)
display(pooled_standardised_effect_sizes.round(4))
print('Saved: standardised_treatment_effect_sizes.csv and pooled_standardised_treatment_effect_sizes.csv')
print('Interpretation: an adjusted treatment contrast expressed in observed outcome SD units; not a standardised predictor coefficient.')
print('CI endpoints divide the regression CI by the observed pooled SD. They treat the SD as fixed and do not include uncertainty in its estimation.')
print('Subgroups have their own denominators; use point effects for direct comparisons across models/categories. The gap is standardised by its own SD.')


## Additional exploratory adjustment for multiple testing

Holm adjustment is applied within separate families of seven model-specific tests for each study and outcome, across all nine Study 2 category tests (three categories × three outcomes), and across seven combined Study 1 recommendation TOST p-values. The additional Study 1 legitimacy and Study 2 legitimacy/gap model tests are included as separate seven-test families. Original p-values are retained. Pooled preregistered inference remains unchanged. Regression confidence intervals remain pointwise; they are not multiplicity-adjusted. An adjusted equivalence conclusion uses the Holm-adjusted combined TOST p-value, not the unadjusted 90% interval alone. This adjustment was added after preregistration.


In [ ]:
# Additional exploratory Holm adjustment. Keep the original p-values and CIs.
from statsmodels.stats.multitest import multipletests

# Families: seven model tests per study and outcome; all nine category tests
# together; and the seven model-specific Study 1 combined TOST p-values.
# Pooled preregistered tests retain their planned inference.
holm_rows = []
def apply_holm_family(frame, mask, p_column, family):
    values = frame.loc[mask, p_column].astype(float)
    if values.empty or values.isna().any() or not values.between(0, 1).all():
        raise ValueError(f'Invalid p-values in {family}')
    reject, adjusted, _, _ = multipletests(values.to_numpy(), alpha=.05, method='holm')
    frame.loc[mask, 'p_holm'] = adjusted
    frame.loc[mask, 'reject_holm_05'] = reject
    frame.loc[mask, 'holm_family'] = family
    frame.loc[mask, 'holm_family_size'] = len(values)
    for idx, original_p, adjusted_p, decision in zip(values.index, values, adjusted, reject):
        row = frame.loc[idx]
        holm_rows.append({
            'family': family, 'family_size': len(values),
            'subgroup': row.get('model_label', row.get('category', '')),
            'outcome': row['outcome'], 'p_unadjusted': float(original_p),
            'p_holm': float(adjusted_p), 'reject_holm_05': bool(decision),
        })

for outcome_h in ['recommendation', 'legitimacy']:
    mask_h = study1_by_model['outcome'].eq(outcome_h)
    apply_holm_family(study1_by_model, mask_h, 'p_unadjusted', f'Study 1 models: {outcome_h}')
for outcome_h in ['recommendation_T5', 'legitimacy_T5', 'gap_T5']:
    mask_h = study2_by_model['outcome'].eq(outcome_h)
    apply_holm_family(study2_by_model, mask_h, 'p_unadjusted', f'Study 2 models: {outcome_h}')
apply_holm_family(category_ancova, pd.Series(True, index=category_ancova.index), 'p',
                  'Study 2 categories: all three outcomes')

# TOST already combines its two one-sided p-values using max(p_lower, p_upper).
# Adjust these seven combined p-values, not fourteen separate component tests.
tost_h = study1_by_model['outcome'].eq('recommendation')
tost_raw_h = study1_by_model.loc[tost_h, 'TOST_p_unadjusted'].astype(float)
tost_reject_h, tost_adjusted_h, _, _ = multipletests(tost_raw_h, alpha=.05, method='holm')
study1_by_model.loc[tost_h, 'TOST_p_holm'] = tost_adjusted_h
study1_by_model.loc[tost_h, 'equivalent_within_margin_holm'] = tost_reject_h
study1_by_model.loc[tost_h, 'TOST_holm_family'] = 'Study 1 model recommendation equivalence'
study1_by_model.loc[tost_h, 'TOST_holm_family_size'] = len(tost_raw_h)
for idx, raw_p, adj_p, decision in zip(tost_raw_h.index, tost_raw_h, tost_adjusted_h, tost_reject_h):
    holm_rows.append({
        'family': 'Study 1 model recommendation equivalence', 'family_size': len(tost_raw_h),
        'subgroup': study1_by_model.loc[idx, 'model_label'], 'outcome': 'recommendation equivalence',
        'p_unadjusted': float(raw_p), 'p_holm': float(adj_p), 'reject_holm_05': bool(decision),
    })

holm_adjusted_tests = pd.DataFrame(holm_rows)
# Refresh exploratory exports; primary estimates/exports remain unchanged.
study1_by_model.to_csv(EXPORTS / 'exploratory_study1_by_model.csv', index=False)
study2_by_model.to_csv(EXPORTS / 'exploratory_study2_by_model.csv', index=False)
category_ancova.to_csv(EXPORTS / 'exploratory_category_turn5_ancova.csv', index=False)
holm_adjusted_tests.to_csv(EXPORTS / 'exploratory_holm_adjusted_tests.csv', index=False)
# Add adjusted p-values to the descriptive standardised-effect export as well.
standardised_effect_sizes['p_holm'] = np.nan
standardised_effect_sizes['holm_family'] = ''
for idx, row in standardised_effect_sizes.iterrows():
    if row['scope'] == 'exploratory_model':
        source_h = study1_by_model if str(row['study']) == '1' else study2_by_model
        match_h = source_h.loc[source_h['model_label'].eq(row['subgroup']) & source_h['outcome'].eq(row['outcome'])]
    elif row['scope'] == 'exploratory_category':
        match_h = category_ancova.loc[category_ancova['category'].eq(row['subgroup']) & category_ancova['outcome'].eq(row['outcome'])]
    else:
        continue
    if len(match_h) != 1:
        raise ValueError('Could not uniquely match an exploratory standardised effect to its Holm-adjusted test.')
    standardised_effect_sizes.loc[idx, 'p_holm'] = float(match_h.iloc[0]['p_holm'])
    standardised_effect_sizes.loc[idx, 'holm_family'] = match_h.iloc[0]['holm_family']
standardised_effect_sizes.to_csv(EXPORTS / 'standardised_treatment_effect_sizes.csv', index=False)
display(holm_adjusted_tests.round(6))
print('Saved Holm-adjusted exploratory tests and refreshed model, category and effect-size CSVs.')
print('Regression 95% CIs remain pointwise, not simultaneous. The 90% TOST intervals show unadjusted equivalence tests; use TOST_p_holm for adjusted conclusions.')
print('This adjustment was added after preregistration. Pooled confirmatory inference and coefficients are unchanged.')


## Additional exploratory appendix: Study 2 numerical endorsement and reversals

This descriptive analysis was added after preregistration. It reports counts, denominators
and percentages by investment category and condition, pooling the seven models equally
under the balanced design. These are the advising models' numerical recommendation
ratings, not independent coding of their written advice or observed investor behaviour.

- **Final numerical endorsement:** Turn 5 recommendation > 50, divided by all complete
  conversations in that category and condition.
- **Reversal, all conversations:** Turn 1 recommendation < 50 AND Turn 5 recommendation
  > 50, divided by all complete conversations in that category and condition.
- **Reversal, initially discouraging conversations:** the same numerator, divided only
  by conversations with Turn 1 recommendation < 50.

Scores exactly equal to 50 are neutral: they are neither endorsement nor initial
discouragement. Zero-event rows are retained. A zero eligible denominator is reported
as NA, not as a zero rate. No significance tests or multiple-testing adjustments are
introduced here; earlier preregistered and exploratory analyses are unchanged.
The CSV exports contain numeric counts, denominators and proportions (0–1), and a
compact formatted table. A conversation-level export makes the classifications auditable.


In [ ]:
# Exploratory descriptive appendix; requires the existing s2 and EXPORTS objects.
# Rebuild paired endpoints directly from turn-level records to detect duplicate rows.
def study2_endorsement_summary(turn_data):
    keys = ['slot_id', 'model_key', 'scenario_id', 'intended_category', 'condition']
    required = keys + ['turn', 'recommendation']
    missing = set(required) - set(turn_data.columns)
    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')
    data = turn_data[required].copy()
    if data[required].isna().any().any():
        raise ValueError('Missing identifiers, turns or recommendation scores.')
    if data.duplicated(['slot_id', 'turn']).any():
        raise ValueError('Duplicate conversation-turn records.')
    if not data['recommendation'].between(0, 100).all():
        raise ValueError('Recommendation scores outside 0–100.')
    consistency = data.groupby('slot_id')[keys[1:]].nunique()
    if not consistency.eq(1).all().all():
        raise ValueError('Conversation metadata change across turns.')
    if not data.groupby('slot_id')['turn'].agg(lambda x: set(x) == {1, 2, 3, 4, 5}).all():
        raise ValueError('Each conversation must contain exactly Turns 1–5.')
    paired = data.pivot(index=keys, columns='turn', values='recommendation')
    paired = paired[[1, 5]].rename(columns={1: 'recommendation_T1', 5: 'recommendation_T5'}).reset_index()
    paired.columns.name = None
    paired['initially_below_50'] = paired['recommendation_T1'].lt(50)
    paired['final_above_50'] = paired['recommendation_T5'].gt(50)
    paired['reversal_below_to_above_50'] = paired['initially_below_50'] & paired['final_above_50']
    paired['initial_equal_50'] = paired['recommendation_T1'].eq(50)
    paired['final_equal_50'] = paired['recommendation_T5'].eq(50)
    rows = []
    # Iterate over observed groups, including those with no endorsement/reversal events.
    for (category, condition), group in paired.groupby(['intended_category', 'condition'], sort=True, observed=True):
        total = len(group)
        eligible = int(group['initially_below_50'].sum())
        final = int(group['final_above_50'].sum())
        reversals = int(group['reversal_below_to_above_50'].sum())
        rows.append({
            'intended_category': category, 'condition': condition,
            'n_conversations': total,
            'n_T5_above_50': final, 'denominator_T5_above_50': total,
            'proportion_T5_above_50': final / total,
            'n_T1_below_50': eligible,
            'n_reversals_T1_below_50_to_T5_above_50': reversals,
            'denominator_reversals_all': total,
            'proportion_reversals_all': reversals / total,
            'denominator_reversals_initially_below_50': eligible,
            'proportion_reversals_initially_below_50': reversals / eligible if eligible else np.nan,
            'n_T1_equal_50': int(group['initial_equal_50'].sum()),
            'n_T5_equal_50': int(group['final_equal_50'].sum()),
        })
    return paired, pd.DataFrame(rows)

endorsement_conversations, endorsement_summary = study2_endorsement_summary(s2)
assert len(endorsement_conversations) == 2940
assert len(endorsement_summary) == 6
assert endorsement_summary.groupby('condition')['n_conversations'].sum().eq(1470).all()
assert endorsement_summary['n_reversals_T1_below_50_to_T5_above_50'].le(endorsement_summary['n_T5_above_50']).all()

def count_percent(numerator, denominator):
    return f'{int(numerator)}/{int(denominator)} ({100 * numerator / denominator:.2f}%)' if denominator else f'{int(numerator)}/0 (NA)'

endorsement_appendix_table = endorsement_summary[['intended_category', 'condition']].copy()
endorsement_appendix_table['Turn 5 > 50: n/N (%)'] = endorsement_summary.apply(
    lambda r: count_percent(r['n_T5_above_50'], r['n_conversations']), axis=1)
endorsement_appendix_table['Reversal: n/all conversations (%)'] = endorsement_summary.apply(
    lambda r: count_percent(r['n_reversals_T1_below_50_to_T5_above_50'], r['n_conversations']), axis=1)
endorsement_appendix_table['Reversal: n/initially below 50 (%)'] = endorsement_summary.apply(
    lambda r: count_percent(r['n_reversals_T1_below_50_to_T5_above_50'], r['n_T1_below_50']), axis=1)

endorsement_summary.to_csv(EXPORTS / 'exploratory_study2_endorsement_reversals_by_category.csv', index=False)
endorsement_appendix_table.to_csv(EXPORTS / 'exploratory_study2_endorsement_reversals_appendix_table.csv', index=False)
endorsement_conversations.to_csv(EXPORTS / 'exploratory_study2_endorsement_reversals_conversations.csv', index=False)
display(endorsement_appendix_table)
print('Exploratory numerical endorsement: >50 at Turn 5. Reversal: <50 at Turn 1 and >50 at Turn 5.')
print('Exact scores of 50 are neutral. Percentages use the denominators shown; NA means no eligible conversations.')
print('Saved numeric summary, formatted appendix table and conversation-level audit CSVs to', EXPORTS)


       intended_category condition Turn 5 > 50: n/N (%) Reversal: n/all conversations (%) Reversal: n/initially below 50 (%)
high-risk but legitimate   neutral        9/315 (2.86%)                     1/315 (0.32%)                      1/224 (0.45%)
high-risk but legitimate  pressure      74/315 (23.49%)                    16/315 (5.08%)                     16/225 (7.11%)
              legitimate   neutral     277/315 (87.94%)                     1/315 (0.32%)                      1/1 (100.00%)
              legitimate  pressure     264/315 (83.81%)                     1/315 (0.32%)                       1/2 (50.00%)
   scam/fraud-indicative   neutral        0/840 (0.00%)                     0/840 (0.00%)                      0/840 (0.00%)
   scam/fraud-indicative  pressure        0/840 (0.00%)                     0/840 (0.00%)                      0/840 (0.00%)

Verified using 2,940 complete Study 2 conversations. Exact 50 is neutral; all counts include zero-event groups.


## Appendix A4 — additional exploratory scenario-specific Study 2 effects

Each scenario is analysed separately using Turn 5 recommendation as the outcome,
condition as the treatment, its own Turn 1 recommendation as the baseline covariate,
and model fixed effects. Model-by-scenario fixed effects reduce to model fixed effects
within a scenario. HC2 robust standard errors and pointwise 95% confidence intervals
are used. Holm adjustment is applied across all 14 scenario recommendation tests.
These analyses were added after preregistration and are exploratory.

The forest plot groups scenarios by their intended category and uses one common point
scale. The exported table includes unadjusted Turn 1 and Turn 5 means in both
conditions, alongside adjusted effects, confidence intervals and raw and adjusted
p-values. Each scenario contributes 210 conversations (105 per condition).
These estimates concern the tested models and stimuli; they do not establish causal
differences between investment categories or independently validate scenario labels.


In [ ]:
# Additional exploratory Study 2 ANCOVA within each of the 14 scenarios.
from statsmodels.stats.multitest import multipletests

scenario_order = ['L1', 'L2', 'L3', 'R1', 'R2', 'R3'] + [f'S{i}' for i in range(1, 9)]
scenario_labels = {s['scenario_id']: s['researcher_label'] for s in manifest['scenarios']}
scenario_rows = []
for scenario_id in scenario_order:
    d = wide.loc[wide['scenario_id'].eq(scenario_id)].copy()
    assert len(d) == 210 and d['model_key'].nunique() == 7
    assert d.groupby('condition').size().eq(105).all()
    fit = smf.ols('recommendation_T5 ~ pressure + recommendation_T1 + C(model_key)', data=d).fit(cov_type='HC2', use_t=True)
    ci = fit.conf_int().loc['pressure']
    row = {'scenario_id': scenario_id, 'scenario_label': scenario_labels[scenario_id],
           'category': d['intended_category'].iloc[0], 'n': len(d),
           'estimate': float(fit.params['pressure']), 'se_HC2': float(fit.bse['pressure']),
           'ci95_low': float(ci.iloc[0]), 'ci95_high': float(ci.iloc[1]),
           'p_unadjusted': float(fit.pvalues['pressure'])}
    for condition in ['neutral', 'pressure']:
        g = d.loc[d['condition'].eq(condition)]
        row[f'n_{condition}'] = len(g)
        for turn in [1, 5]:
            row[f'mean_T{turn}_{condition}'] = float(g[f'recommendation_T{turn}'].mean())
    scenario_rows.append(row)
scenario_ancova = pd.DataFrame(scenario_rows)
reject, adjusted, _, _ = multipletests(scenario_ancova['p_unadjusted'], alpha=.05, method='holm')
scenario_ancova['p_holm'] = adjusted
scenario_ancova['reject_holm_05'] = reject
scenario_ancova['holm_family'] = 'Study 2 scenario recommendation effects'
scenario_ancova['holm_family_size'] = 14
scenario_ancova.to_csv(EXPORTS / 'exploratory_study2_recommendation_by_scenario.csv', index=False)

# Single common point scale, including zero; CIs are pointwise and unadjusted.
fig, ax = plt.subplots(figsize=(11, 7.4))
positions = {}; y = 18
groups = [('Clearly legitimate', ['L1','L2','L3'], '#38688c'),
          ('High-risk but legitimate', ['R1','R2','R3'], '#b95b32'),
          ('Scam-indicative', [f'S{i}' for i in range(1,9)], '#59616a')]
ticks = []; labels = []
for title, ids, color in groups:
    ax.text(-.40, y, title, transform=ax.get_yaxis_transform(), fontweight='bold', fontsize=10)
    y -= 1
    for sid in ids:
        r = scenario_ancova.loc[scenario_ancova.scenario_id.eq(sid)].iloc[0]
        positions[sid] = y
        ax.errorbar(r['estimate'], y, xerr=[[r['estimate']-r['ci95_low']], [r['ci95_high']-r['estimate']]],
                    fmt='o', color=color, capsize=3, markersize=4.5)
        ticks.append(y); labels.append(f"{sid}  {r['scenario_label']}")
        y -= 1
    y -= .8
ax.set_yticks(ticks); ax.set_yticklabels(labels, fontsize=9)
ax.set_ylim(y+.3,19)
ax.axvline(0, color='black', linewidth=.8, linestyle='--')
ax.set_xlabel('Adjusted pressure effect on Turn 5 recommendation (points)', fontsize=10)
ax.grid(axis='x', alpha=.18)
ax.spines[['top','right','left']].set_visible(False)
ax.tick_params(axis='y', length=0)
fig.subplots_adjust(left=.40, right=.98, top=.96, bottom=.11)
fig.savefig(FIGURES / 'Figure_A4_exploratory_scenario_effects.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURES / 'Figure_A4_exploratory_scenario_effects.pdf', bbox_inches='tight')
plt.show()
display(scenario_ancova[['scenario_id','estimate','ci95_low','ci95_high','p_holm',
                        'mean_T1_neutral','mean_T5_neutral','mean_T1_pressure','mean_T5_pressure']].round(3))
print('Exploratory Appendix A4: 14 separate ANCOVAs; model fixed effects, own Turn 1 recommendation, HC2 standard errors.')
print('Holm adjustment across all 14 tests. Confidence intervals are pointwise and unadjusted. Means are unadjusted.')


## Figure outputs

All visualisations are exported automatically in both PNG and PDF format under:

`analysis_exports/figures/`

The statistical specification remains the preregistered one; figures are presentation/diagnostic views of the same data.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=50cf22b4-6086-46b7-bde8-9b24fcd40297' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>